<a href="https://colab.research.google.com/github/emzu/futureIDF/blob/main/0)Download_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

In [1]:
from google.colab import drive
drive.mount('/content/drive')

try:
    !git clone "https://github.com/emzu/futureIDF"
except:
    print("Already cloned")

%cd /content/futureIDF
!git pull

# Load Packages
!pip install -r requirements.txt

Mounted at /content/drive
Cloning into 'futureIDF'...
remote: Enumerating objects: 2211, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 2211 (delta 9), reused 63 (delta 4), pack-reused 2133 (from 2)
Receiving objects: 100% (2211/2211), 281.98 MiB | 26.48 MiB/s, done.
Resolving deltas: 100% (272/272), done.
Updating files: 100% (2100/2100), done.
/content/futureIDF
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.9 MB/s eta 0:00

In [4]:

!git pull

# Import modules
import sys
import importlib
# Ensure workspace root is on sys.path so the local `modules` package can be imported
sys.path.append('..')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import xarray as xr
import glob
import os
import warnings
warnings.filterwarnings('ignore')

from modules import config, data_io, timeseries, plotting, process_rp, geospatial
importlib.reload(timeseries)
importlib.reload(data_io)
importlib.reload(process_rp)
importlib.reload(geospatial)

Already up to date.


<module 'modules.geospatial' from '/content/futureIDF/modules/geospatial.py'>

## Atlas 14

In [ ]:
#!pip install pfdf -i https://code.usgs.gov/api/v4/groups/859/-/packages/pypi/simple

Looking in indexes: https://code.usgs.gov/api/v4/groups/859/-/packages/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.8/240.8 kB 797.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.5/91.5 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pysheds: filename=pysheds-0.4-py3-none-any.whl size=90688 sha256=61ffdc507b13559abe0bc81b584ea02c172f1aba3c4833f0786137e061618c78
  Stored in directory: /root/.cache/pip/wheels/3a/43/fe/f7368d98c408b59b96ce1bee99575571f1640ff4b379be8089
Successfully built pysheds


In [ ]:

counties_gdf = data_io.load_counties()

df = data_io.get_atlas14(counties_gdf)

df.to_parquet('atlas14_24hr_data.parquet', index=False)

Downloaded Beaver: extracted 3 24-hr records
Downloaded Salem: extracted 3 24-hr records
Downloaded Montgomery: extracted 3 24-hr records
Downloaded Poquoson: extracted 3 24-hr records
Downloaded James City: extracted 3 24-hr records
Downloaded Forest: extracted 3 24-hr records
Downloaded Stafford: extracted 3 24-hr records
Downloaded Tioga: extracted 3 24-hr records
Downloaded Surry: extracted 3 24-hr records
Downloaded Dauphin: extracted 3 24-hr records
Downloaded Worcester: extracted 3 24-hr records
Downloaded Wayne: extracted 3 24-hr records
Downloaded Allegany: extracted 3 24-hr records
Downloaded Baltimore: extracted 3 24-hr records
Downloaded Hanover: extracted 3 24-hr records
Downloaded Russell: extracted 3 24-hr records
Downloaded Greene: extracted 3 24-hr records
Downloaded Henry: extracted 3 24-hr records
Downloaded Alexandria: extracted 3 24-hr records
Downloaded Charlottesville: extracted 3 24-hr records
Downloaded King and Queen: extracted 3 24-hr records
Downloaded Wayne

# NOAA CO-OP Station Network

In [ ]:
#Look up station name from station inventory
df = pd.read_csv('/content/futureIDF/data/NOAA COOP/HPD_v02r02_stationinv_c20250701.csv')
coop_stationinv = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_xy(df['Lon'], df['Lat']), crs=4326).to_crs(3857)

#Crop Station Data to MARISA bounds
aoi = gpd.read_file('/content/drive/MyDrive/Research/MARISA_IDF/data/Boundaries/MARISA_domain.shp')
aoi = aoi.to_crs(3857)
coop_stationinv_cbp = coop_stationinv.clip(aoi).to_crs(4326)
for i in range(len(df.columns)):
  coop_stationinv_cbp.rename(columns={i: df.columns[i]}, inplace=True)

#Compute precipitation frequency thresholds
ppf = data_io.get_station_ppf(coop_stationinv_cbp)

## LOCA